# AutoGluon AutoML Tutorial — Multi‑Dataset (Iris, Wine, Adult Income)

Welcome! This tutorial consists of two parts, first you'll learn some basics about AutoML using AutoGluon after which this will be applied on BO algorithm. 

We use **tabular AutoML with [AutoGluon](https://auto.gluon.ai/)** across three classic datasets:
- **Iris** (multiclass classification)
- **Wine** (multiclass classification)
- **Adult Income** (binary classification)

First we'll practice some basics with AutoML:
- Quick starts with `TabularPredictor`
- Train/validation/test splits
- Leaderboards and model diagnostics
- Basic feature importance
- Simple, reproducible experiment structure
- **student exercises** (✓ markers) and **reflection questions** (💭)

After, this will be applied to BO!
- Solve relatively simple problem through BO
- BO parameters are chosen using AutoML
- Visualise different iterations
---

**How to use this notebook**
1) Run the **Setup** cell below to install AutoGluon - most packages are already in the environment installed during the first tutorial, you might have to install AutoGluon by hand.
2) Work through each dataset section. Complete the **✓ Exercises** cells.
3) Answer the **Reflection** questions in the provided markdown cells.

If you’re new to AutoGluon: it automatically **trains, tunes, and ensembles** strong tabular models with minimal code.


## Setup
Installs and imports. Re‑run this cell if your runtime resets.

In [ ]:
# !pip -q install -U pip setuptools wheel #<-- Sometimes fails on building wheels when not present
# !pip -q install -U autogluon #<-- For installing AutoGluon

import os, numpy as np, pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report
from autogluon.tabular import TabularPredictor
pd.set_option('display.max_colwidth', 200)

BASE_DIR = './autogluon_tutorial'  # default local
os.makedirs(BASE_DIR, exist_ok=True)
BASE_DIR


## 1) Iris — Multiclass Classification (Easy)
The Iris dataset has 150 flowers with **four numeric features** and a **3‑class target**. It’s clean and tiny — perfect for a warm‑up.

In [ ]:
## Load Iris data set
iris = load_iris(as_frame=True)
X = iris.frame.copy()
X.rename(columns={'target': 'label'}, inplace=True)
label = 'label'

## Split data set into train and test
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42, stratify=X[label])
X_train.shape, X_test.shape, X[label].nunique()

In [ ]:
save_path = os.path.join(BASE_DIR, 'iris_ag')

## Setup tabular predictor
predictor_iris = TabularPredictor(label=label, problem_type='multiclass', path=save_path)
predictor_iris.fit(
    train_data=X_train,
    time_limit=120,
    presets='medium_quality'
)

## Fit tebular predictors on Iris dataset
leaderboard_iris = predictor_iris.leaderboard(X_test, silent=True)
leaderboard_iris

## Visualisation
Plot the confusion matrix with predictions and have a look at the feature importance. Is feature importance model agnostic?

In [ ]:
pred_test = predictor_iris.predict(X_test)
print('Test accuracy:', (pred_test == X_test[label]).mean())
print('\nClassification report:')
print(classification_report(X_test[label], pred_test))

cm = confusion_matrix(X_test[label], pred_test, labels=pred_test.unique())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pred_test.unique())
plt.figure()
disp.plot(colorbar=False)
plt.title('Iris — Confusion Matrix')
plt.show()


In [ ]:
fi = predictor_iris.feature_importance(X_test)
fi.head(10)

### ✓ Exercise (Iris)
1. Re‑train with a stricter time budget (e.g., `time_limit=30`). How does accuracy change?
2. Try `presets='best_quality'` with a small `time_limit`. Any difference in the top models?
3. (Stretch) Add a synthetic noise column and check feature importance.


In [ ]:
# TODO: Implement your code here
# Example scaffold:
# predictor_iris_fast = TabularPredictor(label='label', problem_type='multiclass', path=os.path.join(BASE_DIR, 'iris_fast')).fit(
#     train_data=X_train,
#     time_limit=30,
#     presets='good_quality'
# )
# predictor_iris_fast.leaderboard(X_test, silent=True)


## 2) Wine — Multiclass Classification (Small/Tabular)
Wine quality (sklearn’s Wine dataset) contains **13 numeric features** describing chemical analysis of wines with **3 classes**.

In [ ]:
## Load wine dataset
wine = load_wine(as_frame=True)
W = wine.frame.copy()
W.rename(columns={'target': 'label'}, inplace=True)
label = 'label'

## Train test split
W_train, W_test = train_test_split(W, test_size=0.2, random_state=42, stratify=W[label])
W_train.shape, W_test.shape, W[label].nunique()

In [ ]:
## Train autoML for Wine data set
save_path = os.path.join(BASE_DIR, 'wine_ag')
predictor_wine = TabularPredictor(label=label, problem_type='multiclass', path=save_path)
predictor_wine.fit(
    train_data=W_train,
    time_limit=10,
    presets='medium_quality'
)
leaderboard_wine = predictor_wine.leaderboard(W_test, silent=True)
leaderboard_wine

## Visualisation
Plot the confusion matrix with predictions and have a look at the feature importance. Is feature importance model agnostic?

In [ ]:

pred_test_w = predictor_wine.predict(W_test)
print('Test accuracy:', (pred_test_w == W_test[label]).mean())
print('\nClassification report:')
print(classification_report(W_test[label], pred_test_w))

cm = confusion_matrix(W_test[label], pred_test_w, labels=pred_test_w.unique())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pred_test_w.unique())
plt.figure()
disp.plot(colorbar=False)
plt.title('Wine — Confusion Matrix')
plt.show()

In [ ]:
## Analyse feature importance per
fi_wine = predictor_wine.feature_importance(W_test)
fi_wine.head(10)

#Q: Is feature importance model specific?

### ✓ Exercise (Wine)
1. Increase `time_limit` and compare leaderboards. Which models benefit most?
2. Try `presets='good_quality_faster_inference_only_refit'`. Does inference speed change?
3. (Stretch) Manually create a train/val split (`tuning_data`) and monitor overfitting.


In [ ]:
# TODO: Your Wine experiments here
## Example scaffold: 
# predictor_wine_fast = TabularPredictor(label='label', problem_type='multiclass', path=os.path.join(BASE_DIR, 'wine_fast')).fit(
#     train_data=W_train,
#     time_limit=60,
#     presets='good_quality_faster_inference_only_refit'
# ) 
# predictor_wine_fast.leaderboard(W_test, silent=True)


## 3) Adult Income — Binary Classification (Realistic)
The **Adult** dataset predicts whether income exceeds $50K/yr. It contains **mixed numeric & categorical features** and mild missingness.

We’ll fetch from **OpenML** (no manual download needed in Colab).

In [ ]:
import pandas as pd
import urllib.request, io

df_adult=pd.read_csv("adult.csv")

# Standardize column names
df_adult.columns = [c.strip().replace(' ', '_').replace('-', '_').lower() for c in df_adult.columns]

# Identify label column variants commonly found
possible_labels = ['income', 'class', '>50k']
label = None
for c in possible_labels:
    if c in df_adult.columns:
        label = c
        break
if label is None:
    # Heuristic: last column is often label
    label = df_adult.columns[-1]
print('Label column:', label)

# Train/test split (stratified)
train_adult, test_adult = train_test_split(df_adult, test_size=0.2, random_state=42, stratify=df_adult[label])
train_adult.shape, test_adult.shape, train_adult[label].value_counts(normalize=True)[:3]

In [ ]:
## Fit tabular prediction on income data
save_path = os.path.join(BASE_DIR, 'adult_ag')
predictor_adult = TabularPredictor(label=label, problem_type='binary', path=save_path, eval_metric='roc_auc')
predictor_adult.fit(
    train_data=train_adult,
    time_limit=60,
    presets='good_quality'
)
leaderboard_adult = predictor_adult.leaderboard(test_adult, silent=True)
leaderboard_adult.head(15)

## Visualisation
Plot the confusion matrix with predictions and have a look at the feature importance. Is feature importance model agnostic?

In [ ]:
pred_proba = predictor_adult.predict_proba(test_adult)
pred_label = predictor_adult.predict(test_adult)
print('Test ROC AUC:', predictor_adult.evaluate(test_adult))
print('\nClassification report:')
print(classification_report(test_adult[label], pred_label))

cm = confusion_matrix(test_adult[label], pred_label)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
plt.figure()
disp.plot(colorbar=False)
plt.title('Adult Income — Confusion Matrix')
plt.show()


In [ ]:
fi_adult = predictor_adult.feature_importance(test_adult)
fi_adult.head(15)

### ✓ Exercise (Adult)
1. Change the evaluation metric to `accuracy` or `f1` and compare.
2. Add `excluded_model_types=['KNN']` (or others) — does it affect results/speed?
3. (Stretch) Use `refit_full=True` after finding the best model, then re‑evaluate.


In [ ]:
# TODO: Your Adult experiments here
## Example scaffold:
# predictor_adult_alt = TabularPredictor(label=label, problem_type='binary', path=os.path.join(BASE_DIR, 'adult_alt'), eval_metric='accuracy').fit(
#     train_data=train_adult,
#     time_limit=300,
#     presets='good_quality',
#     excluded_model_types=['KNN']
# )
# predictor_adult_alt.leaderboard(test_adult, silent=True)


## 4) Compare Results Across Datasets
Aggregate the top models and metrics to reflect on how dataset properties influence AutoML outcomes.

In [ ]:
def top_model_summary(name, leaderboard_df, metric_col='score_val'):
    df = leaderboard_df.copy()
    # Try common metric columns in case names differ
    for c in ['score_val', 'score_test', 'score', 'validation_score']:
        if c in df.columns:
            metric_col = c
            break
    top = df.sort_values(by=metric_col, ascending=False).head(1)
    top['dataset'] = name
    return top[['dataset','model','fit_time','pred_time_val','pred_time_val_median', metric_col]]

compare_rows = []
try:
    compare_rows.append(top_model_summary('Iris', leaderboard_iris))
except Exception:
    pass
try:
    compare_rows.append(top_model_summary('Wine', leaderboard_wine))
except Exception:
    pass
try:
    compare_rows.append(top_model_summary('Adult', leaderboard_adult))
except Exception:
    pass

if compare_rows:
    comp = pd.concat(compare_rows, ignore_index=True)
    comp
else:
    print('Run the training sections first to compare results.')
